In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.6028571428571428, 10: 0.5949999999999998, 20: 0.5928571428571429, 30: 0.6014285714285714, 40: 0.61, 50: 0.5992857142857141, 60: 0.6078571428571429, 70: 0.6107142857142858, 80: 0.6057142857142855, 90: 0.6092857142857143, 100: 0.6000000000000001, 110: 0.5999999999999999, 120: 0.5921428571428572, 130: 0.605, 140: 0.6028571428571426, 150: 0.5957142857142856, 160: 0.5907142857142859, 170: 0.6150000000000001, 180: 0.6050000000000001, 190: 0.6092857142857141, 200: 0.6092857142857143, 210: 0.6035714285714285, 220: 0.6007142857142856, 230: 0.5921428571428574, 240: 0.6064285714285713, 250: 0.6214285714285714, 260: 0.6171428571428572, 270: 0.6166666666666665, 280: 0.6250000000000001, 290: 0.6093749999999999, 300: 0.6161290322580646}
{0: 0.007348979591836735, 10: 0.00992142857142857, 20: 0.010770408163265306, 30: 0.010105102040816326, 40: 0.008757142857142857, 50: 0.008660204081632651, 60: 0.00931326530612245, 70: 0.007688775510204082, 80: 0.008253061224489794, 90: 0.0076102040816326535, 100